# 017a — Parked extract: AvgSA sanity check & site MAFC

These are one-off exploratory checks pulled out of `017-disagg_imls_for_msa_stripes.ipynb`
so the production notebook stays focused on IML selection. **This is a loose extract**: the
cells below depend on objects defined in nb 017's earlier sections (`ANALYSIS_ROOT`, `cfg`,
`site_specific_fcs`, `IML_GRID`, `site_pc_curve`, `GROUPS`, `site_vb`, `hazard`/`hcs_*`, ...),
so run nb 017 up to the selection section first, then run these in the same kernel.

- **§1 Sanity check** — AvgSA(0-3s) IDA vs. converted-SA IDA fragility comparison.
- **§2 Site MAFC** — approximate mean annual frequency of collapse per site.


## 1. Sanity check


### Sanity Check

I have performed a sanity check on the AvgSA values to try and find out if there is an error - or the buildings as designed just have a large capacity relative to the seismic hazard level of the sites that they are designed for.

1. I have checked the approximate AvgSA[0,3] calculated from the downloaded ESHM20 2475yr RTP hazard maps against the 2475 yr AvgSA values obtained directly from the reduced hazard model that I have run myself in OpenQuake. A visual comparison seems to indicate that the values of AvgSA from both sources are in the ballpark of each other. There doesn't seem to to be any error larger enough to acount for the 100-500% factor needed to cause a sufficient number of collapses. Based on this I think I can rule out an underestimation of the hazard. Maybe I should do a hazard run for a site at 2475yr for a couple of sites at a few SA(T1) periods to see how much the results differ from my model compared to the original ESHM20 model. --> I haven't done this yet and it would be a good safety check.

2. The second check on AvgSA[0,3] is making sure that the capacity side is correctly represented. The fragility curves of the reference structures were determined from IDA on SA(T1) and then converted to AvgSA[0,3] based on the record scaling factors that cause collapse. A separate notebook (C:\Users\clemettn\Documents\phd\notebooks\03_WP1_ground_motion_set\checks_and_tests\avgSA_conversion.ipynb) has been created to manually check the AvgSA[0,3] conversion. Comparing these values with the values from the fragilty curve conversions (014-sdof_validation_and_fragilities.ipynb, §2.7.1), the values are identical and it appears that the IM conversion of the fragility curves is valid. To double check we can rerun some of the IDAs but this time using the AvgSA[0,3] as the IM. The results below show only a very small difference (<2%) in the median collapse intensity. This I think can be attributed to the scaling and conversion process - the effects of which can be gleaned by looking at the emipirical cdf points. The pattern is very similar, however there are differences. The AvgSA IDA has vertical lines of points where failure occurs at the sam iml for several records. In the case of the converted SA results we don't see this in the same way because two records that cause collapse at the same SA IML will likely have different AvgSA values due to the different spectral shapes. From these results I have to conclude that the conversion / choice of IML is not the cause for the very high capacity

3. The third check that we can do is also on the capacity side. Currently a valid failure is recorded when the structure exceeds 20% interstorey drift. This is very large and to be honest 10% drift could also be considered a collapse. However, we can check how sensitive the results are to this assumption by running a couple of AvgSA-based IDAs on the MDOF systems with 10% and 20% drift limits to see how different the fragility curves are. 

#### Comparison of AvgSA IDA vs. Converted SA IDA

In [ ]:
# compare the fragiltiy curves from teh AvgSA_03 IDA and the SA(T1) IDA for
# the 3s_cbf_dc2_10 SDOF structure
 
# Import the AvgSA_03 fragility curve directly from avgsa_03 ida
with open(ANALYSIS_ROOT / "3s_cbf_dc2_10_sdof/ida_femap695_avgsa03/collapse_fragility.json", "r") as file:
    avgsa_03_fc = json.load(file)

# Import the AvgSA_03 fragility curve directly from avgsa_03 ida v2 (different hunt parameters)
with open(ANALYSIS_ROOT / "3s_cbf_dc2_10_sdof/ida_femap695_avgsa03_v2/collapse_fragility.json", "r") as file:
    avgsa_03_2_fc = json.load(file)

# Import the AvgSA_03 fragility curve converted from the SA(T1) ida
with open(cfg["proc_data"]["sdof_fragility_curves"] / "sdof_fragility_curves.pickle", "rb") as file:
    avgsa_03_fcs_converted = pickle.load(file)
avgsa_03_fc_conv = avgsa_03_fcs_converted["3s_cbf_dc2_10"].asdict()


In [ ]:
print(f"AvgSA IDA: Median = {avgsa_03_fc["median"]:.3f}   Dispersion = {avgsa_03_fc["dispersion"]:.3f}")
print(f"AvgSA IDA: Median = {avgsa_03_2_fc["median"]:.3f}   Dispersion = {avgsa_03_2_fc["dispersion"]:.3f}")
print(f"SA IDA:    Median = {avgsa_03_fc_conv["median"]:.3f}   Dispersion = {avgsa_03_fc_conv["dispersion"]:.3f}")

diff_pc = (avgsa_03_fc["median"] - avgsa_03_fc_conv["median"]) / avgsa_03_fc["median"] * 100
print(f"Diff in the median = {diff_pc:.3f}%")

imls = np.linspace(0.01, 0.5*9810)

avgsa_03_fit = lognorm.cdf(imls, s=avgsa_03_fc["dispersion"], scale=avgsa_03_fc["median"])
avgsa_03_conv_fit = lognorm.cdf(imls, s=avgsa_03_fc_conv["dispersion"], scale=avgsa_03_fc_conv["median"])

In [ ]:
g = 9810   # m/s

fig, ax = plt.subplots()
ax.plot(np.array(avgsa_03_fc["efc"][0]) / 9810, avgsa_03_fc["efc"][1], ls="", marker="o", mfc="b", mec="k", label="AvgSA IDA - ecdf")
ax.plot(imls / 9810, avgsa_03_fit, color="b", label="AvgSA IDA - fit")
ax.plot(np.array(avgsa_03_fc_conv["efc"][0]) / 9810, avgsa_03_fc_conv["efc"][1], ls="", marker="o", mfc="g", mec="k", label="SA IDA - ecdf")
ax.plot(imls / 9810, avgsa_03_conv_fit, color="g", label="SA IDA - fit")

ax.grid(ls="-.",color="0.8")
ax.set_xlim(0.1, 0.5)
ax.set_xlabel("AvgSA[0,3]  [g]")
ax.set_ylabel("P[collapse]")

leg = ax.legend()
frame = leg.get_frame()
frame.set_edgecolor("k")
frame.set_facecolor("white")
frame.set_alpha(1.0)
plt.tight_layout()

## 2. Site-specific mean annual frequency of collapse (approx.)


In [ ]:
from hazrisk.risk import numerical_mafe
# get the fragility curves for each site specific structure.
# Interpolated based on the vb_coeff of the site-specific design and the 
# vb_coeff of the site-independent reference designs

site_fc_curves = {}
for site_range, _ in GROUPS:
    for site_id in site_range:
        colour = CMAP(NORM(site_vb[site_id]))
        site_fc_curves[site_id] = np.column_stack([IML_GRID, site_pc_curve(site_vb[site_id])])

# integrate the fragility with respect to hazard to get the MAFC
mafc = {}
for site, hc in hcs_03_mean.items():
    idx_max = np.where(hc[:, 1] <= 0.0)[0][0]
    imls = np.logspace(min(np.log10(hc[:idx_max, 0])), max(np.log10(hc[:idx_max, 0])), 100)

    # interpolate the fragility and hazard are defined at the same IMLs
    H = np.exp(np.interp(np.log(imls), np.log(hc[:idx_max, 0]), np.log(hc[:idx_max, 1])))
    Pc = (np.interp(imls, site_fc_curves[site][:, 0], site_fc_curves[site][:, 1]))
    
    # calculate the MAFC
    mafc[site] = numerical_mafe(H, Pc)

mafcs = list(mafc.values())

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(mafcs, ls="", marker="o", mfc="b", mec="k", alpha=0.7)
ax.hlines(2e-4, -1, 60, color="r")
ax.grid(ls="-.", color="0.8")
ax.set_ylabel("Mean Ann. Freq. Collapse, MAFC [1/yr]")
ax.set_xlabel("Site ID")
ax.set_ylim(1e-7, 5e-4)
ax.set_xlim(-1, 60)
ax.set_title("Estimate of MAFC for European Casestudy Sites")
ax.annotate("EC8:Annex F limit", (0.5, 2.1e-4))